# 🚀 Qwen2-VL-2B Fine-Tuning with PEFT & Google Drive Integration
**Autonomous Web Navigation Agent Fine-Tuning on 100% Real Ground-Truth Cases**

Ensure GPU acceleration is enabled: **Runtime** -> **Change runtime type** -> **T4 GPU**.

### Step 1: Install Dependencies
Installs pinned, stable versions of `transformers==4.48.3`, `bitsandbytes>=0.50.0`, `peft==0.14.0`, and `accelerate==1.3.0` compatible with CUDA 13 and Python 3.13.

In [ ]:
# Install pinned, compatible stack for Qwen2-VL 4-Bit QLoRA on CUDA 13
!pip install -q "bitsandbytes>=0.50.0" accelerate==1.3.0 transformers==4.48.3 peft==0.14.0 qwen-vl-utils torchvision sentencepiece tiktoken

import transformers, peft, accelerate, qwen_vl_utils
import bitsandbytes as bnb
print("=" * 60)
print(f"✅ transformers : {transformers.__version__}")
print(f"✅ peft         : {peft.__version__}")
print(f"✅ bitsandbytes : {bnb.__version__}")
print(f"✅ accelerate   : {accelerate.__version__}")
print("=" * 60)
print("All dependencies successfully installed and verified!")


### Step 3: Mount Google Drive & Extract Dataset
Mounts Google Drive, automatically detects `vlm_dataset.zip` (in `VLM_TEST_COLAB/` or root `MyDrive/`), and extracts to local high-speed storage.

In [ ]:
import os
import zipfile
from google.colab import drive

# Mount Google Drive
try:
    drive.mount("/content/drive")
    print("Google Drive successfully mounted.")
except Exception as e:
    print(f"Drive note: {e}")

# Target candidate paths in Google Drive
candidate_zips = [
    "/content/drive/MyDrive/VLM_TEST_COLAB/vlm_dataset.zip",
    "/content/drive/MyDrive/vlm_dataset.zip",
    "/content/drive/MyDrive/live_vlm_dataset.zip"
]
zip_path = next((p for p in candidate_zips if os.path.exists(p)), None)
extract_path = "/content/live_vlm_dataset"

if zip_path and os.path.exists(zip_path):
    print(f"Found dataset archive at {zip_path}. Extracting...")
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")
elif os.path.exists(extract_path):
    print(f"Found existing extracted dataset at {extract_path}")
else:
    print("Warning: Dataset zip not found in Drive. Cloning fallback from GitHub...")
    !git clone --depth 1 https://github.com/Janakiram-2005/vlm-testing.git /content/vlm-testing
    extract_path = "/content/vlm-testing/live_vlm_dataset"

print(f"Active dataset directory: {extract_path}")


### Step 4: Fine-tune Qwen2-VL-2B (Clean Model Setup)
Sets up 4-bit NormalFloat (NF4) quantization on GPU and attaches LoRA adapters to attention and MLP projection layers.

In [ ]:
import os
import sys
os.environ["USE_TF"] = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Defensive hook to ensure bitsandbytes multi-backend compatibility
import transformers.utils
if not hasattr(transformers.utils, "is_bitsandbytes_multi_backend_available"):
    try:
        from transformers.utils.import_utils import is_bitsandbytes_multi_backend_available
        transformers.utils.is_bitsandbytes_multi_backend_available = is_bitsandbytes_multi_backend_available
    except Exception:
        transformers.utils.is_bitsandbytes_multi_backend_available = lambda: True

import torch
import bitsandbytes as bnb
import accelerate

try:
    from transformers import Qwen2TokenizerFast as AutoTokenizer
except Exception:
    from transformers import AutoTokenizer

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2-VL-2B-Instruct"

# 4-bit Quantization configuration (Float16 compute for Nvidia T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading processor and 4-bit quantized model (this will take a few moments)...")
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

# LoRA configuration targeting linear attention & MLP layers
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("✅ Model preparation complete! Ready for Step 4.5 training.")


### Step 4.5: Prepare Dataset and Run Training Loop
Loads the real verified ground-truth web actions and trains the Qwen2-VL adapter with step-by-step loss reporting.

In [ ]:
import os
import json
import torch
from PIL import Image

possible_jsonls = [
    os.path.join(extract_path, "verified_true_cases.jsonl"),
    os.path.join(extract_path, "dataset.jsonl"),
    os.path.join(extract_path, "live_vlm_dataset", "verified_true_cases.jsonl"),
    os.path.join(extract_path, "live_vlm_dataset", "dataset.jsonl"),
    "/content/live_vlm_dataset/dataset.jsonl",
    "/content/live_vlm_dataset/live_vlm_dataset/dataset.jsonl"
]
data_file = next((p for p in possible_jsonls if os.path.exists(p)), None)
if not data_file:
    raise FileNotFoundError(f"Could not find dataset.jsonl in {extract_path}")

possible_img_dirs = [
    os.path.join(os.path.dirname(data_file), "images"),
    os.path.join(extract_path, "images"),
    "/content/live_vlm_dataset/images",
    "/content/live_vlm_dataset/live_vlm_dataset/images"
]
images_dir = next((p for p in possible_img_dirs if os.path.exists(p) and len(os.listdir(p)) > 0), None)

with open(data_file, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f if line.strip()]

print(f"✅ Loaded {len(samples)} real web action records from {data_file}")
print(f"✅ Found {len(os.listdir(images_dir))} screenshot images in {images_dir}")

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
epochs = 3
print("\n" + "=" * 60)
print("🚀 Starting Fine-Tuning on Real Web Actions...")
print("=" * 60)

for epoch in range(epochs):
    epoch_loss = 0.0
    valid_count = 0
    for idx, item in enumerate(samples):
        img_name = item.get("image_file", "")
        img_path = os.path.join(images_dir, img_name)
        if not os.path.exists(img_path):
            continue
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            continue

        messages = [
            {"role": "system", "content": "You are an expert autonomous web navigation agent. Output THOUGHT and ACTION with exact bounding box coordinates."},
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": item.get("instruction", "Click target element.")}]},
            {"role": "assistant", "content": [{"type": "text", "text": item.get("ground_truth", "")}]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")
        inputs["labels"] = inputs["input_ids"].clone()

        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        valid_count += 1

        if (idx + 1) % 20 == 0 or idx == len(samples) - 1:
            print(f"Epoch [{epoch+1}/{epochs}] | Step [{idx+1}/{len(samples)}] | Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / max(1, valid_count)
    print(f"🏁 Epoch [{epoch+1}/{epochs}] Completed - Avg Loss: {avg_loss:.4f}\n")

print("🎉 Fine-tuning completed successfully! Ready for Step 5 export.")


### Step 5: Save Adapter and Export to Google Drive
Exports the LoRA adapter weights, packages into zip, saves to Google Drive (`MyDrive/` and `MyDrive/VLM_TEST_COLAB/`), and triggers a browser download.

In [ ]:
import os
import shutil
from google.colab import files

output_dir = "/content/healvlm_colab_adapter"
os.makedirs(output_dir, exist_ok=True)

# Save fine-tuned adapter weights and processor
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print(f"✅ Adapter saved locally to {output_dir}")

# Package adapter into zip
zip_base = "/content/healvlm_colab_adapter"
archive_path = shutil.make_archive(zip_base, "zip", output_dir)
print(f"✅ Packaged adapter into {archive_path}")

# Permanent backup to Google Drive
if os.path.exists("/content/drive/MyDrive"):
    shutil.copy2(archive_path, "/content/drive/MyDrive/healvlm_colab_adapter.zip")
    print("✅ Backup permanently saved to /content/drive/MyDrive/healvlm_colab_adapter.zip")
    if os.path.exists("/content/drive/MyDrive/VLM_TEST_COLAB"):
        shutil.copy2(archive_path, "/content/drive/MyDrive/VLM_TEST_COLAB/healvlm_colab_adapter.zip")
        print("✅ Backup also saved to /content/drive/MyDrive/VLM_TEST_COLAB/healvlm_colab_adapter.zip")

# Trigger browser download
print("\nTriggering direct browser download...")
try:
    files.download(archive_path)
except Exception as e:
    print(f"Browser download notice: {e}")
